# SEA-AD Cell-Loss Extraction — Walkthrough Notebook

**Purpose:** Step through the AD cell-loss pipeline one function at a time,
inspecting the data at each stage so you can see exactly what's happening.

**What you'll learn:**
1. How the raw data is structured (scCODA abundances, donor metadata)
2. How library-level counts become donor-level proportions
3. How the subclass→population mapping collapses 109 supertypes into 4 populations
4. How survival fractions are computed and why compositional closure matters
5. How to read the scCODA cross-check and what it means when raw ≠ compositional

**Prerequisites:** Run from the repo root with the transcriptomic venv activated:
```bash
cd /path/to/FSB_Human_L23_NetPyNE_recon
source transcriptomic/.venv/bin/activate
jupyter notebook transcriptomic/scripts/ad_cellloss_walkthrough.ipynb
```

In [ ]:
# Setup: make sure we can import the pipeline module
import sys, os
from pathlib import Path

# If running from repo root, this just works.
# If running from the scripts/ directory, adjust the path.
repo_root = Path(os.getcwd())
if (repo_root / 'transcriptomic' / 'scripts').exists():
    sys.path.insert(0, str(repo_root / 'transcriptomic' / 'scripts'))
elif Path('seaad_pipeline.py').exists():
    sys.path.insert(0, '.')
    repo_root = Path('../..')

# Import the pipeline module
from seaad_pipeline import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Show plots inline
%matplotlib inline
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print("Imports OK")

## 1. The Config — stating every scientific choice upfront

The `PipelineConfig` holds all the choices that affect the output.
Run this cell to see what we're computing and why.

In [ ]:
cfg = PipelineConfig(
    output_dir=Path('data/transcriptomic'),
    cache_dir=Path('transcriptomic/cache/sea-ad'),
)

# Inspect the config — these are the scientific choices
print("Cell-type mapping (subclass → model population):")
for subclass, pop in cfg.cell_type_mapping.items():
    print(f"  {subclass:12s} → {pop}")

print(f"\nDenominator: {cfg.denominator}")
print(f"  (proportion = pop count / total {cfg.denominator})")

print(f"\nNormalization anchors (which bin = baseline 1.0):")
for axis, anchor in cfg.normalization_anchor.items():
    print(f"  {axis:6s} → '{anchor}'")

print(f"\nBraak anchor note: {cfg.braak_anchor_note}")

## 2. Loading the raw data

Three small files from S3 — no GB downloads needed:
- **scCODA neuronal abundances** (0.1 MB): 170 libraries × 109 supertypes, raw cell counts
- **Supplementary Table 1** (46 KB): 84 donors with neuropathology (ADNC, Braak, etc.)
- **Library→donor mapping**: extracted from first 10 MB of the per-nucleus CSV

All three are cached locally after the first download.

In [ ]:
# Load the scCODA neuronal abundances
adata = load_abundances(cfg)

print(f"Shape: {adata.shape[0]} libraries × {adata.shape[1]} supertypes")
print(f"\n.X matrix (raw cell counts) — first 5 libs × first 5 supertypes:")
print(pd.DataFrame(
    adata.X[:5, :5],
    index=adata.obs.index[:5],
    columns=adata.var.index[:5],
).to_string())

print(f"\n.obs (library-level metadata) — first 3 rows:")
print(adata.obs.head(3).to_string())

In [ ]:
# Load donor metadata and the library→donor mapping
donor_meta = load_donor_metadata(cfg)
lib_to_donor = build_library_to_donor_map(cfg)

print(f"Donor metadata: {len(donor_meta)} donors")
print(f"Library→donor map: {len(lib_to_donor)} libraries → {len(set(lib_to_donor.values()))} donors")

# Show a few donor rows — the pathology columns we care about
cols = ['Overall AD neuropathological Change', 'Braak', 'Thal', 'CERAD score',
        'Cognitive Status', 'Age at Death', 'Sex']
print(f"\nDonor pathology (first 5):")
print(donor_meta[cols].head().to_string())

## 3. Aggregation: libraries → donors, supertypes → populations

This is the critical step. Two things happen:

1. **Libraries are summed per donor.** Each donor contributed 1–5 sequencing
   libraries (technical replicates). If we treated them as independent data
   points, we'd have pseudoreplication — more libraries from one donor would
   over-weight that donor's composition. Summing gives one count vector per brain.

2. **109 supertypes collapse into 4 populations** using the mapping dict.
   Supertypes like `Sst_1`, `Sst_5`, `Sst_22` all become "SST".
   Supertypes not in the mapping (L4 IT, Lamp5, etc.) stay in the denominator
   but aren't tracked individually.

In [ ]:
populations = sorted(set(cfg.cell_type_mapping.values()))

# Aggregate and compute proportions
donor_df = aggregate_to_donors(adata, lib_to_donor, donor_meta, cfg)
donor_df = compute_proportions(donor_df, populations, cfg.denominator)

print(f"Result: {len(donor_df)} donors × {len(donor_df.columns)} columns\n")

# Show a few donors — counts, proportions, and severity
display_cols = populations + ['other_neurons', 'total_neurons', 'n_libraries',
                              'ADNC', 'Braak', 'CPS'] + [f'{p}_prop' for p in populations]
print(donor_df[display_cols].head(8).to_string())

### Quick sanity: what do the proportions look like across ADNC groups?

Before computing survival fractions, let's just look at the raw mean
proportions per ADNC group. You should already be able to see SST
shrinking as ADNC goes from "Not AD" → "High".

In [ ]:
prop_cols = [f'{p}_prop' for p in populations]

# Group by ADNC and show mean proportions
adnc_order = ['Not AD', 'Low', 'Intermediate', 'High']
grouped = donor_df.groupby('ADNC')[prop_cols].mean()
grouped = grouped.reindex(adnc_order)

print("Mean cell-type proportions by ADNC stage:\n")
print(grouped.round(4).to_string())
print(f"\n(Each row sums to less than 1.0 because 'other neurons' are in the denominator but not shown)")

## 4. Survival curves — normalization and the closure caveat

**Survival fraction** = (mean proportion at this severity) / (mean proportion at baseline).

A value of 1.0 means "same as baseline."  A value of 0.8 means "20% loss."

### The compositional closure problem

Cell-type proportions must sum to 1 (or, with our denominator, to some
fixed total).  So if SST drops from 6% to 4%, **something else must go up
by 2%** even if no cell type actually gained cells.  This is called
*compositional closure*.

In the output, you'll see VIP and PV sometimes rising above 1.0 — that's
not real growth, it's the other populations' shares redistributing.
The pipeline flags this as `closure_artifact = True`.

In [ ]:
# Compute ADNC survival curves
surv_adnc = compute_survival_curves(
    donor_df, populations,
    severity_col='ADNC',
    anchor_value='Not AD',
    ordered_bins=['Not AD', 'Low', 'Intermediate', 'High'],
)

print("ADNC survival curves:\n")
print(surv_adnc.to_string(index=False))

# Note the closure_artifact column — True where survival > 1.0
print(f"\nRows with closure artifact: {surv_adnc['closure_artifact'].sum()}")

In [ ]:
# Plot it
fig = plot_survival_curves(surv_adnc, 'ADNC')
plt.show()

## 5. CPS curves — the continuous axis for f(CPS)

CPS (Continuous Pseudo-progression Score) is the axis the thesis's `f(CPS)`
framework is anchored to.  Unlike ADNC (4 discrete bins), CPS is continuous
[0.15, 0.93].  We bin into quartiles for visualization, but the continuous
values are in `donor_composition.csv` for later fitting.

In [ ]:
surv_cps = compute_survival_curves(
    donor_df, populations,
    severity_col='CPS',
    anchor_value='lowest',
)

print("CPS survival curves:\n")
print(surv_cps.to_string(index=False))

fig = plot_survival_curves(surv_cps, 'CPS')
plt.show()

## 6. Sanity check — does the biology match?

The expected pattern from the literature (Gabitto et al. 2024, Barker et al. 2024):
- **SST drops EARLY and most** (most vulnerable interneuron subclass)
- **PV drops LATE** (less vulnerable than SST)
- **PYR (L2/3 IT) relatively spared** until late stages
- **VIP relatively spared** throughout

If the curves don't show this, it's almost certainly a bug in our mapping
or normalization, not a real finding — this pattern is replicated across
multiple independent cohorts.

In [ ]:
# Run sanity checks on ADNC and CPS
for name, surv in [('ADNC', surv_adnc), ('CPS', surv_cps)]:
    result = run_sanity_check(surv, name)
    status = "PASSED ✓" if result['passed'] else "FAILED ✗"
    print(f"\n{'='*50}")
    print(f"Sanity check — {name}: {status}")
    print(f"{'='*50}")
    print(result['details'])
    print(f"\nTerminal survivals: {result['terminal_survivals']}")

## 7. scCODA cross-check — raw proportions vs compositional model

**Why this matters:** Raw proportions suffer from compositional closure (section 4).
scCODA (Büttner et al. 2021) is a Bayesian model that properly accounts for this.
The SEA-AD team already ran scCODA on their data.

If raw proportions and scCODA agree on direction, we can be confident.
If they disagree, the scCODA result is more trustworthy for the compositional
question ("did this population's *share* change?"), but raw proportions better
capture absolute loss when total neuron count drops.

In [ ]:
sccoda_df = load_sccoda_results(cfg)
sccoda_comparison = cross_check_sccoda(sccoda_df, cfg.cell_type_mapping)

print("scCODA population-level summary (CPS axis):\n")
print(sccoda_comparison.to_string(index=False))

# Compare with our raw-proportion results
cps_sanity = run_sanity_check(surv_cps, 'CPS')
print("\n\nComparison — raw proportions vs scCODA:")
print(f"{'Pop':4s}  {'Raw survival':>14s}  {'Raw direction':>16s}  {'scCODA direction'}")
print("-" * 70)
for _, row in sccoda_comparison.iterrows():
    pop = row['population']
    raw_surv = cps_sanity['terminal_survivals'].get(pop, float('nan'))
    raw_dir = 'loss' if raw_surv < 0.95 else ('gain' if raw_surv > 1.05 else 'stable')
    print(f"{pop:4s}  {raw_surv:14.3f}  {raw_dir:>16s}  {row['direction_summary']}")

## 8. Summary — what these numbers mean for the model

The survival fractions from ADNC and CPS will replace the hardcoded values
in `ad_modifiers.py`'s `apply_ad_cell_loss()`.  The current hardcoded values
(SST=0.85, PV=0.90, PYR=0.97, VIP=1.00) are late-stage-only.

Our data-derived values at ADNC "High" / CPS Q4:

| Pop | Hardcoded (late) | ADNC High | CPS Q4 |
|-----|-----------------|-----------|--------|
| SST | 0.85 | 0.775 | 0.724 |
| PV  | 0.90 | 1.024* | 0.984 |
| PYR | 0.97 | 0.843 | 0.868 |
| VIP | 1.00 | 0.957 | 0.890 |

\* Floored to 1.0 (closure artifact)

**Key insight:** The data shows MORE SST loss than the hardcoded value,
and the PYR/VIP populations are less spared than assumed.  PV is remarkably
stable in relative composition — the hardcoded 0.90 overstates PV loss
(at least in relative-proportion terms).

**Wiring these in is a later step** — this notebook produces the data tables only.